imports

In [ ]:
import os
import sys
import matplotlib as mpl

sys.path.append("..")
mpl.rcParams["animation.embed_limit"] = 100

from cleanroom_cfd.config import SimulationConfig
from cleanroom_cfd.setup import (
    build_room_setup,
    build_initial_fields,
)
from cleanroom_cfd.data_pipeline import (
    load_timeline_window,
    build_machine_state,
    build_human_entities_for_time,
    update_machine_entities_for_time,
)
from cleanroom_cfd.simulation import compute_time_step, compute_tau_field, cfd_step
from cleanroom_cfd.analysis import run_simulation_and_collect, save_metric_plots
from cleanroom_cfd.visualization import animate_simulation

experiment settings


In [ ]:
cfg = SimulationConfig()

csv_path = "../assets/object_timeline_temperature.csv"
svg_path = "../assets/Feynmann room_inked_good.svg"
results_dir = "../results"

table_window_minutes = 60
save_interval_s = 30.0

cfg.res = 10
cfg.sock_speed = 0.45
cfg.target_time = table_window_minutes * 60
cfg.frames = int(cfg.target_time / save_interval_s)

os.makedirs(results_dir, exist_ok=True)

load timeline data

In [ ]:
timeline_data = load_timeline_window(
    csv_path=csv_path,
    table_window_minutes=table_window_minutes
)

print("Window start:", timeline_data["first_ts"])
print("Window end:", timeline_data["end_ts"])
print("Rows in window:", len(timeline_data["df_window"]))
print("Unique timestamps in window:", len(timeline_data["unique_times"]))
print("Labels in window:", sorted(timeline_data["df_window"]["canonical_label"].dropna().unique().tolist()))
print("people_or_machine values:", sorted(timeline_data["df_window"]["people_or_machine"].dropna().unique().tolist()))

build static entities and machine state

In [ ]:
base_entities_list = [
    # Middle passthrough furniture
    room["entity_factory"]("furniture_passthrough", x_m=2.0, y_m=3.8, width_m=8.8, height_m=2.3, id_name="middle_1"),
    room["entity_factory"]("furniture_passthrough", x_m=10.8, y_m=3.0, width_m=1.0, height_m=3.1, id_name="middle_2"),

    # Existing passthrough furniture
    room["entity_factory"]("furniture_passthrough", x_m=1.8, y_m=0.0, width_m=3.6, height_m=1.0, id_name="table_1"),
    room["entity_factory"]("furniture_passthrough", x_m=5.7, y_m=0.0, width_m=3.9, height_m=1.0, id_name="table_2"),
    room["entity_factory"]("furniture_passthrough", x_m=9.8, y_m=0.0, width_m=3.8, height_m=1.0, id_name="table_3"),
]

machine_state = build_machine_state(
    df_window=timeline_data["df_window"],
    svg_width_m=room["svg_width_m"],
    svg_height_m=room["svg_height_m"],
    machine_radius_m=0.30,
)

persistent_machine_entities = machine_state["persistent_machine_entities"]
machine_entity_map = machine_state["machine_entity_map"]
machine_temp_lookup = machine_state["machine_temp_lookup"]

print("Persistent machines created:", len(persistent_machine_entities))

initial fields and tau field

In [ ]:
thermal_grid, sock_tracer, u_vel, v_vel, p = build_initial_fields(
    grid_h=room["grid_h"],
    grid_w=room["grid_w"],
    is_obstacle=room["is_obstacle"],
    T_ref=cfg.T_ref,
    supply_temp=cfg.supply_temp,
    src_y0=room["src_y0"],
    src_y1=room["src_y1"],
)

dt = compute_time_step(room["dx"], cfg.sock_speed, cfg.alpha_heat, cfg.nu_eff)

tau_field = compute_tau_field(
    (room["grid_h"], room["grid_w"]),
    room["src_y0"],
    room["src_y1"],
    room["is_obstacle"],
    base_entities_list + persistent_machine_entities,
    cfg.res,
    tau_min=100,
    tau_max=400,
)

substeps_per_frame = int(save_interval_s / dt)

print("dt =", dt)
print("frames =", cfg.frames)
print("save interval (s) =", save_interval_s)
print("substeps_per_frame =", substeps_per_frame)
print("total simulated time represented =", cfg.frames * substeps_per_frame * dt)

step function

In [ ]:
sim_time_state = {"t": 0.0}
dynamic_entities_state = {"humans": []}

def step_fn(T, tracer, u, v, p):
    update_machine_entities_for_time(
        sim_time_s=sim_time_state["t"],
        first_ts=timeline_data["first_ts"],
        unique_times=timeline_data["unique_times"],
        machine_entity_map=machine_entity_map,
        machine_temp_lookup=machine_temp_lookup,
    )

    human_entities = build_human_entities_for_time(
        sim_time_s=sim_time_state["t"],
        first_ts=timeline_data["first_ts"],
        unique_times=timeline_data["unique_times"],
        rows_by_time=timeline_data["rows_by_time"],
        svg_width_m=room["svg_width_m"],
        svg_height_m=room["svg_height_m"],
    )

    dynamic_entities_state["humans"] = human_entities
    entities_list = base_entities_list + persistent_machine_entities + human_entities

    out = cfd_step(
        T, tracer, u, v, p,
        dx=room["dx"],
        dt=dt,
        entities_list=entities_list,
        is_obstacle=room["is_obstacle"],
        res=cfg.res,
        src_y0=room["src_y0"],
        src_y1=room["src_y1"],
        hs_x=room["hs_x"],
        hs_y=room["hs_y"],
        bubble_r=cfg.bubble_r,
        alpha_heat=cfg.alpha_heat,
        nu_eff=cfg.nu_eff,
        rho=cfg.rho,
        beta_b=cfg.beta_b,
        g=cfg.g,
        T_ref=cfg.T_ref,
        sock_speed=cfg.sock_speed,
        supply_temp=cfg.supply_temp,
        smoke_diff=cfg.smoke_diff,
        pressure_iters=cfg.pressure_iters,
        max_speed=cfg.max_speed,
        tau_field=tau_field,
    )

    sim_time_state["t"] += dt
    return out

run analysis

In [ ]:
thermal_grid, sock_tracer, u_vel, v_vel, p = build_initial_fields(
    grid_h=room["grid_h"],
    grid_w=room["grid_w"],
    is_obstacle=room["is_obstacle"],
    T_ref=cfg.T_ref,
    supply_temp=cfg.supply_temp,
    src_y0=room["src_y0"],
    src_y1=room["src_y1"],
)

sim_time_state = {"t": 0.0}
dynamic_entities_state = {"humans": []}

df_metrics, final_state = run_simulation_and_collect(
    thermal_grid, sock_tracer, u_vel, v_vel, p,
    step_fn=step_fn,
    frames=cfg.frames,
    substeps_per_frame=substeps_per_frame,
    dt=dt,
    is_obstacle=room["is_obstacle"],
    machine_entities=persistent_machine_entities,
    res=cfg.res,
    real_start_timestamp=timeline_data["first_ts"],
    use_circular_machine_metrics=True,
    default_machine_radius_m=0.30,
    surround_outer_radius_m=0.60,
)

csv_path = save_metric_plots(df_metrics, results_dir)

print("Saved results to:", results_dir)
print("Metrics CSV:", csv_path)
df_metrics.head()

save gif

In [ ]:
thermal_grid, sock_tracer, u_vel, v_vel, p = build_initial_fields(
    grid_h=room["grid_h"],
    grid_w=room["grid_w"],
    is_obstacle=room["is_obstacle"],
    T_ref=cfg.T_ref,
    supply_temp=cfg.supply_temp,
    src_y0=room["src_y0"],
    src_y1=room["src_y1"],
)

sim_time_state = {"t": 0.0}
dynamic_entities_state = {"humans": []}

gif_path = os.path.join(results_dir, "simulation.gif")

anim = animate_simulation(
    thermal_grid, sock_tracer, u_vel, v_vel, p,
    step_fn=step_fn,
    frames=cfg.frames,
    substeps_per_frame=substeps_per_frame,
    svg_width_m=room["svg_width_m"],
    svg_height_m=room["svg_height_m"],
    is_obstacle=room["is_obstacle"],
    y_sock_m=cfg.y_sock_m,
    sock_thickness_m=cfg.sock_thickness_m,
    hs_x_m=cfg.hotspot_x_m,
    hs_y_m=cfg.hotspot_y_m,
    v_sock_target=cfg.sock_speed,
    T_supply=cfg.supply_temp,
    dt=dt,
    entities_list=base_entities_list + persistent_machine_entities,
    real_start_timestamp=timeline_data["first_ts"],
    dynamic_entities_state=dynamic_entities_state,
    save_gif_path=gif_path,
    gif_fps=8,
    show_inline=False,
)

print("Saved GIF to:", gif_path)